# Toxic Colors Model
### Based on Fernandez et al. 2018

In [1]:
# Package loading
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd

keras.utils.set_random_seed(42)

In [3]:
all_images = []
img_directory = 'molecule_images/'

# Runs from 1 to 1547(the greatest numbered img) and will throw errors but continue on missing file names.
for i in range(1, 1548):
    try:
        img = tf.io.read_file('molecule_images/BACE_' + str(i) + '.png')
        all_images.append(tf.io.decode_png(img))
    except:
        pass

## Read in data to extract target features and create train, validation, test split

In [4]:
# Extract target variable
bace_data = pd.read_csv('bace.csv')
pIC = bace_data['pIC50']

# Split into training and test/validation sets
X_train, X_tval, y_train, y_tval = train_test_split(all_images, pIC, test_size = 0.2, random_state = 42)

# Split test and validation sets
X_val, X_test, y_val, y_test = train_test_split(X_tval, y_tval, test_size = 0.5, random_state = 42)

# Convert X's to tensors to avoid later errors
X_train = tf.convert_to_tensor(X_train)
X_test = tf.convert_to_tensor(X_test)
X_val = tf.convert_to_tensor(X_val)

## Model Construction

In [5]:
inputs = keras.Input(shape = (400, 400, 3))

x = layers.Resizing(180, 180)(inputs)
x = layers.Rescaling(1./255)(x)
x = layers.Conv2D(filters = 12, kernel_size = 16, activation = "relu")(x)
x = layers.Dropout(0.4)(x)
x = layers.MaxPool2D(pool_size = (2, 2))(x)
x = layers.Flatten()(x)
x = layers.Dense(40, activation = "relu")(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(1, activation = "linear")(x)
model = keras.Model(inputs = inputs, outputs = outputs)

model.compile(loss = "mse",
              optimizer = "adam",
              metrics = ["mae"])


In [8]:
model.fit(X_train, y_train, epochs = 100, validation_data = (X_val, y_val), batch_size = 128)

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - loss: 4.5018 - mae: 1.7223 - val_loss: 1.9577 - val_mae: 1.1849
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step - loss: 4.2757 - mae: 1.6641 - val_loss: 1.9520 - val_mae: 1.1824
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 89ms/step - loss: 4.3008 - mae: 1.6822 - val_loss: 1.9498 - val_mae: 1.1815
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 87ms/step - loss: 4.3098 - mae: 1.6749 - val_loss: 1.9449 - val_mae: 1.1793
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step - loss: 4.2864 - mae: 1.6716 - val_loss: 1.9446 - val_mae: 1.1792
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step - loss: 4.5387 - mae: 1.7146 - val_loss: 1.9433 - val_mae: 1.1786
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step - loss: 4.3277 - mae: 1.6558 - val_loss: 1.9426 - val_mae: 1.1782
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 95ms/step - loss: 4.5701 - mae: 1.7240 - val_loss: 1.9399 - val_mae: 1.1770
Epoch 9/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - lo

Interesting that this model basically doesn't improve, but shows better loss on the validation data then the training data.

In [9]:
model.evaluate(X_test, y_test)

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 322ms/step - loss: 1.7706 - mae: 1.1042


[1.7706398963928223, 1.1042020320892334]